#  Molecular Fingerprints

Molecules can be represented in many ways.  This tutorial introduces a type of representation called a "molecular fingerprint".  It is a very simple representation that often works well for small drug-like molecules.

## Colab

This tutorial and the rest in this sequence can be done in Google colab. If you'd like to open this notebook in colab, you can use the following link.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepchem/deepchem/blob/master/examples/tutorials/Molecular_Fingerprints.ipynb)




In [1]:
!pip install --pre deepchem

INFO: pip is looking at multiple versions of deepchem to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of deepchem to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.0/610.0 kB 3.1 MB/s eta 0:00:00


We can now import the `deepchem` package to play with.

In [2]:
import deepchem as dc
dc.__version__

/usr/local/lib/python3.13/dist-packages/deepchem/feat/smiles_tokenizer.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.


'2.6.0.dev'

# What is a Fingerprint?

Deep learning models almost always take arrays of numbers as their inputs.  If we want to process molecules with them, we somehow need to represent each molecule as one or more arrays of numbers.

Many (but not all) types of models require their inputs to have a fixed size.  This can be a challenge for molecules, since different molecules have different numbers of atoms.  If we want to use these types of models, we somehow need to represent variable sized molecules with fixed sized arrays.

Fingerprints are designed to address these problems.  A fingerprint is a fixed length array, where different elements indicate the presence of different features in the molecule.  If two molecules have similar fingerprints, that indicates they contain many of the same features, and therefore will likely have similar chemistry.

DeepChem supports a particular type of fingerprint called an "Extended Connectivity Fingerprint", or "ECFP" for short.  They also are sometimes called "circular fingerprints".  The ECFP algorithm begins by classifying atoms based only on their direct properties and bonds.  Each unique pattern is a feature.  For example, "carbon atom bonded to two hydrogens and two heavy atoms" would be a feature, and a particular element of the fingerprint is set to 1 for any molecule that contains that feature.  It then iteratively identifies new features by looking at larger circular neighborhoods.  One specific feature bonded to two other specific features becomes a higher level feature, and the corresponding element is set for any molecule that contains it.  This continues for a fixed number of iterations, most often two.

Let's take a look at a dataset that has been featurized with ECFP.

In [4]:
#download rdkit
%pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 37.3 MB/s eta 0:00:00


In [5]:
from rdkit import Chem
print(Chem.rdBase.rdkitVersion)

2026.09.1


In [9]:
import sys
import deepchem as dc
import rdkit
import numpy as np

print("Python:", sys.version.split()[0])
print("DeepChem:", dc.__version__)
print("RDKit:", rdkit.__version__)
print("NumPy:", np.__version__)

Python: 3.13.16
DeepChem: 2.6.0.dev
RDKit: 2026.09.1
NumPy: 2.1.3


In [10]:
%pip uninstall -y deepchem
%pip install --no-cache-dir deepchem==2.8.0

Found existing installation: deepchem 2.6.0.dev20211026183818
Uninstalling deepchem-2.6.0.dev20211026183818:
  Successfully uninstalled deepchem-2.6.0.dev20211026183818
ERROR: Ignored the following versions that require a different python version: 2.6.0.dev20220112162333 Requires-Python >=3.7,<3.10; 2.6.0.dev20220114040838 Requires-Python >=3.7,<3.10; 2.6.0.dev20220118010103 Requires-Python >=3.7,<3.10; 2.6.0.dev20220118135955 Requires-Python >=3.7,<3.10; 2.6.1 Requires-Python >=3.7,<3.10; 2.6.1.dev20220118213325 Requires-Python >=3.7,<3.10; 2.6.1.dev20220121040525 Requires-Python >=3.7,<3.10; 2.6.1.dev20220121191633 Requires-Python >=3.7,<3.10; 2.6.1.dev20220127040719 Requires-Python >=3.7,<3.10; 2.6.1.dev20220127201742 Requires-Python >=3.7,<3.10; 2.6.1.dev20220208165000 Requires-Python >=3.7,<3.10; 2.6.1.dev20220210194849 Requires-Python >=3.7,<3.10; 2.6.1.dev20220211203047 Requires-Python >=3.7,<3.10; 2.6.1.dev20220218060535 Requires-Python >=3.7,<3.10; 2.6.1.dev20220218194027 Requ

In [1]:
import sys
print(sys.version)

3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


In [2]:
%pip install deepchem==2.8.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 64.2 MB/s eta 0:00:00


In [3]:
import deepchem as dc
import rdkit
import numpy as np

print("DeepChem:", dc.__version__)
print("RDKit:", rdkit.__version__)
print("NumPy:", np.__version__)

wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.


DeepChem: 2.8.0
RDKit: 2026.09.1
NumPy: 2.0.2


In [4]:
#download
tasks, datasets, transformers = dc.molnet.load_tox21(
    featurizer='ECFP'
)

train_dataset, valid_dataset, test_dataset = datasets
print(train_dataset)

流式输出内容被截断，只能显示最后 5000 行内容。
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNING: please use MorganGenerator
[01:00:00] DEPRECATION WARNIN

<DiskDataset X.shape: (np.int64(6258), np.int64(1024)), y.shape: (np.int64(6258), np.int64(12)), w.shape: (np.int64(6258), np.int64(12)), task_names: ['NR-AR' 'NR-AR-LBD' 'NR-AhR' ... 'SR-HSE' 'SR-MMP' 'SR-p53']>


In [5]:
print("Training samples:", len(train_dataset))
print("Validation samples:", len(valid_dataset))
print("Test samples:", len(test_dataset))

print("Feature shape:", train_dataset.X.shape)
print("Tasks:", tasks)

Training samples: 6258
Validation samples: 782
Test samples: 783
Feature shape: (6258, 1024)
Tasks: ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


In [6]:
tasks, datasets, transformers = dc.molnet.load_tox21(
    featurizer='Raw',
    reload=False
)

train_dataset, valid_dataset, test_dataset = datasets

print(train_dataset)

[01:01:27] WARNING: not removing hydrogen atom without neighbors
[01:01:28] Explicit valence for atom # 8 Al, 6, is greater than permitted
    rdkit.Chem.rdmolfiles.CanonicalRankAtoms(NoneType)
did not match C++ signature:
    CanonicalRankAtoms(RDKit::ROMol mol, bool breakTies=True, bool includeChirality=True, bool includeIsotopes=True, bool includeAtomMaps=True, bool includeChiralPresence=False)
[01:01:28] Explicit valence for atom # 3 Al, 6, is greater than permitted
    rdkit.Chem.rdmolfiles.CanonicalRankAtoms(NoneType)
did not match C++ signature:
    CanonicalRankAtoms(RDKit::ROMol mol, bool breakTies=True, bool includeChirality=True, bool includeIsotopes=True, bool includeAtomMaps=True, bool includeChiralPresence=False)
[01:01:28] Explicit valence for atom # 4 Al, 6, is greater than permitted
    rdkit.Chem.rdmolfiles.CanonicalRankAtoms(NoneType)
did not match C++ signature:
    CanonicalRankAtoms(RDKit::ROMol mol, bool breakTies=True, bool includeChirality=True, bool includeIso

<DiskDataset X.shape: (np.int64(6258),), y.shape: (np.int64(6258), np.int64(12)), w.shape: (np.int64(6258), np.int64(12)), task_names: ['NR-AR' 'NR-AR-LBD' 'NR-AhR' ... 'SR-HSE' 'SR-MMP' 'SR-p53']>


The feature array `X` has shape (6264, 1024).  That means there are 6264 samples in the training set.  Each one is represented by a fingerprint of length 1024.  Also notice that the label array `y` has shape (6264, 12): this is a multitask dataset.  Tox21 contains information about the toxicity of molecules.  12 different assays were used to look for signs of toxicity.  The dataset records the results of all 12 assays, each as a different task.

Let's also take a look at the weights array.

In [7]:
train_dataset.w

array([[1.0448833 , 1.03636706, 1.12518597, ..., 1.05583151, 1.17490775,
        1.05294456],
       [1.0448833 , 1.03636706, 1.12518597, ..., 1.05583151, 1.17490775,
        1.05294456],
       [1.0448833 , 1.03636706, 1.12518597, ..., 1.05583151, 0.        ,
        1.05294456],
       ...,
       [1.0448833 , 1.03636706, 1.12518597, ..., 1.05583151, 6.71729958,
        1.05294456],
       [1.0448833 , 1.03636706, 1.12518597, ..., 0.        , 1.17490775,
        1.05294456],
       [1.0448833 , 1.03636706, 1.12518597, ..., 0.        , 0.        ,
        1.05294456]])

Notice that some elements are 0.  The weights are being used to indicate missing data.  Not all assays were actually performed on every molecule.  Setting the weight for a sample or sample/task pair to 0 causes it to be ignored during fitting and evaluation.  It will have no effect on the loss function or other metrics.

Most of the other weights are close to 1, but not exactly 1.  This is done to balance the overall weight of positive and negative samples on each task.  When training the model, we want each of the 12 tasks to contribute equally, and on each task we want to put equal weight on positive and negative samples.  Otherwise, the model might just learn that most of the training samples are non-toxic, and therefore become biased toward identifying other molecules as non-toxic.

# Training a Model on Fingerprints

Let's train a model.  In earlier tutorials we use `GraphConvModel`, which is a fairly complicated architecture that takes a complex set of inputs.  Because fingerprints are so simple, just a single fixed length array, we can use a much simpler type of model.

In [8]:
model = dc.models.MultitaskClassifier(n_tasks=12, n_features=1024, layer_sizes=[1000])

`MultitaskClassifier` is a simple stack of fully connected layers.  In this example we tell it to use a single hidden layer of width 1000.  We also tell it that each input will have 1024 features, and that it should produce predictions for 12 different tasks.

Why not train a separate model for each task?  We could do that, but it turns out that training a single model for multiple tasks often works better.  We will see an example of that in a later tutorial.

Let's train and evaluate the model.

In [10]:
print("X dtype:", train_dataset.X.dtype)
print("y dtype:", train_dataset.y.dtype)
print("w dtype:", train_dataset.w.dtype)

X dtype: object
y dtype: float64
w dtype: float64


In [12]:
import deepchem as dc

tasks, datasets, transformers = dc.molnet.load_tox21(
    featurizer='ECFP',
    reload=False
)

train_dataset, valid_dataset, test_dataset = datasets

print("X shape:", train_dataset.X.shape)
print("X dtype:", train_dataset.X.dtype)
print("First sample type:", type(train_dataset.X[0]))

流式输出内容被截断，只能显示最后 5000 行内容。
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNING: please use MorganGenerator
[01:03:31] DEPRECATION WARNIN

X shape: (6258, 1024)
X dtype: float64
First sample type: <class 'numpy.ndarray'>


In [13]:
import numpy as np

model.fit(train_dataset, nb_epoch=10)
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)
print('training set score:', model.evaluate(train_dataset, [metric], transformers))
print('test set score:', model.evaluate(test_dataset, [metric], transformers))

training set score: {'roc_auc_score': np.float64(0.9586638932508285)}
test set score: {'roc_auc_score': np.float64(0.682324446918258)}


Not bad performance for such a simple model and featurization.  More sophisticated models do slightly better on this dataset, but not enormously better.

# Congratulations! Time to join the Community!

Congratulations on completing this tutorial notebook! If you enjoyed working through the tutorial, and want to continue working with DeepChem, we encourage you to finish the rest of the tutorials in this series. You can also help the DeepChem community in the following ways:

## Star DeepChem on [GitHub](https://github.com/deepchem/deepchem)
This helps build awareness of the DeepChem project and the tools for open source drug discovery that we're trying to build.

## Join the DeepChem Gitter
The DeepChem [Gitter](https://gitter.im/deepchem/Lobby) hosts a number of scientists, developers, and enthusiasts interested in deep learning for the life sciences. Join the conversation!

## Citing This Tutorial
If you found this tutorial useful please consider citing it using the provided BibTeX.

In [ ]:
@manual{Intro4,
 title={Molecular Fingerprints},
 organization={DeepChem},
 author={Ramsundar, Bharath},
 howpublished = {\url{https://github.com/deepchem/deepchem/blob/master/examples/tutorials/Molecular_Fingerprints.ipynb}},
 year={2021},
}